In [ ]:
def search_bakeries(item, location="Delhi"):
    print(f"  🔍 Searching bakeries for '{item}' in {location}...")
    return [
        {"name": "Sweet Layers",    "price": 449, "rating": 4.8, "delivery_min": 45},
        {"name": "Cakewala",        "price": 389, "rating": 4.1, "delivery_min": 70},
        {"name": "Brown Sugar Co.", "price": 499, "rating": 4.5, "delivery_min": 55},
        {"name": "Bake & Smile",    "price": 520, "rating": 4.7, "delivery_min": 40},
        {"name": "Flour Power",     "price": 410, "rating": 3.9, "delivery_min": 80},
        {"name": "The Cake Room",   "price": 480, "rating": 4.6, "delivery_min": 50},
    ]


def check_reviews(bakery_name):
    print(f"  ⭐ Checking reviews for {bakery_name}...")
    reviews_db = {
        "Sweet Layers":    {"highlights": "Rich, moist, loved by kids",    "complaints": "None"},
        "Cakewala":        {"highlights": "Affordable",                     "complaints": "Dry texture reported"},
        "Brown Sugar Co.": {"highlights": "Premium quality, good packing",  "complaints": "Slightly expensive"},
        "Bake & Smile":    {"highlights": "Fast delivery, tasty",           "complaints": "Over budget"},
        "Flour Power":     {"highlights": "Budget friendly",                "complaints": "Average taste"},
        "The Cake Room":   {"highlights": "Great presentation, fresh",      "complaints": "Slow delivery"},
    }
    return reviews_db.get(bakery_name, {"highlights": "No data", "complaints": "No data"})


def calculate_budget(bakeries, max_budget):
    print(f"  💰 Filtering bakeries within ₹{max_budget}...")
    within_budget = [b for b in bakeries if b["price"] <= max_budget]
    if not within_budget:
        return None
    return min(within_budget, key=lambda x: x["price"])


print("✅ Tools registered:")
print("   • search_bakeries(item, location)")
print("   • check_reviews(bakery_name)")
print("   • calculate_budget(bakeries, max_budget)")

In [ ]:
import re


user_query = "Find me the best chocolate cake under ₹500."

print("📥 User Query:", user_query)
print()


def parse_intent(query):
    """
    Extracts: item, budget, urgency, location
    In a real agent → replace this with an LLM call.
    """
    intent = {}

    # Extract item
    item_match = re.search(r'best (.+?) under', query, re.IGNORECASE)
    intent["item"] = item_match.group(1).strip() if item_match else "cake"

    # Extract budget
    budget_match = re.search(r'[₹Rs\.]+\s?(\d+)', query)
    intent["budget_max"] = int(budget_match.group(1)) if budget_match else 500

    # Detect urgency
    urgency_keywords = ["tonight", "now", "asap", "urgent", "today", "party"]
    intent["urgent"] = any(kw in query.lower() for kw in urgency_keywords)

    # Default location
    intent["location"] = "Delhi"

    # Constraints summary
    intent["constraints"] = [
        f"price ≤ ₹{intent['budget_max']}",
        "high rating preferred",
        "fast delivery" if intent["urgent"] else "normal delivery"
    ]

    return intent


intent = parse_intent(user_query)

print("🔄 Parsing intent...")
print()
print("✅ Extracted Intent:")
print(f"   item         = {intent['item']}")
print(f"   budget_max   = ₹{intent['budget_max']}")
print(f"   urgent       = {intent['urgent']}")
print(f"   location     = {intent['location']}")
print(f"   constraints  = {intent['constraints']}")

---
## 🔧 STEP 3 — Agent Calls the Tools

> 👉 **"The agent decides which tools to call and in what order"**
>
> This is the **agentic loop** — the heart of any AI agent.

In [ ]:
print("⚙️  Agent is working...")
print("─" * 50)

# ── Tool Call 1: Search Bakeries ─────────────────────────────
bakeries = search_bakeries(intent["item"], intent["location"])
print(f"   → Found {len(bakeries)} bakeries\n")

# ── Tool Call 2: Check Reviews for top candidates ────────────
# Agent is smart — it only checks reviews for bakeries within budget
within_budget = [b for b in bakeries if b["price"] <= intent["budget_max"]]
reviews = {}
for bakery in within_budget:
    reviews[bakery["name"]] = check_reviews(bakery["name"])
print()

# ── Tool Call 3: Find Cheapest Within Budget ─────────────────
cheapest = calculate_budget(bakeries, intent["budget_max"])
print(f"   → Cheapest within budget: {cheapest['name']} @ ₹{cheapest['price']}")

print("─" * 50)
print("\n✅ All tool calls complete. Moving to scoring...")

---
## ⚖️ STEP 4 — Score & Compare

> 👉 **"The agent doesn't just pick the cheapest — it balances price, rating, and delivery speed"**
>
> This is called a **scoring function** with weights.

In [ ]:
# ── Scoring weights ──────────────────────────────────────────
# Price          : lower is better  → weight 0.35
# Rating         : higher is better → weight 0.40
# Delivery Speed : faster is better → weight 0.25

def score_bakeries(bakeries, max_budget):
    """
    Scores each bakery. Over-budget ones are eliminated.
    Higher score = better option.
    """
    eligible = [b for b in bakeries if b["price"] <= max_budget]

    min_price    = min(b["price"]        for b in eligible)
    max_price    = max(b["price"]        for b in eligible)
    min_delivery = min(b["delivery_min"] for b in eligible)
    max_delivery = max(b["delivery_min"] for b in eligible)
    min_rating   = min(b["rating"]       for b in eligible)
    max_rating   = max(b["rating"]       for b in eligible)

    scored = []
    for b in eligible:
        price_score    = (1 - (b["price"]        - min_price)    / (max_price    - min_price    + 1)) * 35
        rating_score   =      (b["rating"]        - min_rating)   / (max_rating   - min_rating   + 0.001) * 40
        delivery_score = (1 - (b["delivery_min"] - min_delivery) / (max_delivery - min_delivery + 1)) * 25
        total          = round(price_score + rating_score + delivery_score, 1)
        scored.append({**b, "score": total})

    return sorted(scored, key=lambda x: x["score"], reverse=True)


ranked = score_bakeries(bakeries, intent["budget_max"])

# Show eliminated ones too
over_budget = [b for b in bakeries if b["price"] > intent["budget_max"]]

print("📊 Bakery Rankings (within budget):")
print(f"{'Rank':<5} {'Bakery':<18} {'Price':>7} {'Rating':>7} {'Delivery':>10} {'Score':>7}")
print("─" * 60)
for i, b in enumerate(ranked, 1):
    tag = "  ← 🏆 BEST" if i == 1 else ""
    print(f"{i:<5} {b['name']:<18} ₹{b['price']:>6} {b['rating']:>7} {b['delivery_min']:>8} min {b['score']:>7}{tag}")

print()
print("❌ Eliminated (over budget):")
for b in over_budget:
    print(f"   {b['name']:<18} ₹{b['price']}  →  exceeds ₹{intent['budget_max']}")

---
## 🏆 STEP 5 — Final Recommendation

> 👉 **"The agent gives a structured, justified answer — not just a text reply"**

In [ ]:
best         = ranked[0]
best_reviews = reviews.get(best["name"], {})

print("=" * 52)
print("  🛒 SMART SHOPPING AGENT — RECOMMENDATION")
print("=" * 52)
print(f"  🎂 Item         : {intent['item'].title()}")
print(f"  🏅 Best Bakery  : {best['name']}")
print(f"  💰 Price        : ₹{best['price']}  (Budget: ₹{intent['budget_max']} ✓)")
print(f"  ⭐ Rating        : {best['rating']} / 5.0")
print(f"  🚚 Delivery     : {best['delivery_min']} minutes")
print(f"  👍 Highlights   : {best_reviews.get('highlights', 'N/A')}")
print(f"  ⚠️  Complaints   : {best_reviews.get('complaints', 'None')}")
print(f"  📈 Agent Score  : {best['score']} / 100")
print("=" * 52)
print()
print("💬 Agent says:")
print(f'   "I recommend the "{best["name"]}" chocolate cake at ₹{best["price"]}.')
print(f'    It has the best balance of price, rating ({best["rating"]}★),')
print(f'    and delivery speed ({best["delivery_min"]} min). Customers say:')
print(f'    {best_reviews.get("highlights", "")}. Enjoy your party!"')

---
## 🚀 BONUS — Full Agent in One Function

> 👉 **"This is how you wrap everything into a single agent call"**
>
> Try changing the query below and re-run!

In [ ]:
def run_shopping_agent(query: str):
    """End-to-end Smart Shopping Agent."""

    print(f"\n📥 Query: {query}")
    print("─" * 50)

    # Step 1: Parse intent
    intent   = parse_intent(query)

    # Step 2 & 3: Call tools
    bakeries = search_bakeries(intent["item"], intent["location"])
    reviews  = {b["name"]: check_reviews(b["name"])
                for b in bakeries if b["price"] <= intent["budget_max"]}
    _        = calculate_budget(bakeries, intent["budget_max"])

    # Step 4: Score
    ranked   = score_bakeries(bakeries, intent["budget_max"])
    best     = ranked[0]

    # Step 5: Output
    print("─" * 50)
    print(f"✅ Best Bakery  : {best['name']}")
    print(f"   Price        : ₹{best['price']}  (Budget ₹{intent['budget_max']} ✓)")
    print(f"   Rating       : {best['rating']} ⭐")
    print(f"   Delivery     : {best['delivery_min']} min")
    print(f"   Agent Score  : {best['score']} / 100")
    print("─" * 50)


# ── Try different queries! ───────────────────────────────────
run_shopping_agent("Find me the best chocolate cake under ₹500.")

---
## 📝 Summary — What We Learned

| Concept | What it means | In this notebook |
|---|---|---|
| **Tool** | A function the agent can call | `search_bakeries`, `check_reviews`, `calculate_budget` |
| **Intent Parsing** | Understanding the user query | `parse_intent()` |
| **Agentic Loop** | Agent calls tools step by step | STEP 3 |
| **Scoring Function** | Comparing options with weights | `score_bakeries()` |
| **Structured Output** | Final justified answer | STEP 5 |

---

### 🔗 Real-World Upgrades
- Replace `search_bakeries()` → **Zomato / Swiggy API**
- Replace `check_reviews()` → **Google Places Reviews API**
- Replace `parse_intent()` → **Claude / GPT LLM call**
- Add memory → agent remembers your taste preferences
- Add more tools → discount coupons checker, allergy filter, calorie counter

> 🎯 **That's the power of AI Agents — they don't just talk, they act!**